# Typing

In [2]:
from typing import (
    Any, Callable, Coroutine, Dict, List, Optional, Tuple, TypeVar, Union, Protocol, Generic, Annotated, ParamSpec
)

In [ ]:
T = TypeVar('T')
# Arguments ('T', bound=class) -> only 1 type, and restricted to that type and its subtype
# Argumetns ('T', str, bytes) -> can only be one of these types

def get_first_element(items: List[T]) -> T:
    if not items:
        raise ValueError("list empty is invalid")
    return items[0]



In [8]:
def add(a: int, b: int) -> int:
    return a + b

def subtract(a: int, b: int) -> int:
    return a - b

def apply_operation(x: int, y: int, operation: Callable[[int, int], int]) -> int:
    return operation(x, y)

res = apply_operation(2, 3, subtract)
res

-1

In [9]:
def get_formatter(format_type: str) -> Callable[[str], str]:
    if format_type == "upper":
        def to_upper(text: str) -> str:
            return text.upper()
        return to_upper
    elif format_type == "lower":
        def to_lower(text: str) -> str:
            return text.lower()
        return to_lower
    else:
        raise ValueError("Invalid format_type")
    
funct = get_formatter('upper')
res = funct("who are u")
res

'WHO ARE U'

In [ ]:
_CALLBACKS: Dict[str, Callable[[], None]] = {} 

def register_callback(callback_id: str, callback_function: Callable[[], None]) -> None:
    print(f"Registering callback: {callback_id}")
    _CALLBACKS[callback_id] = callback_function

In [11]:
R = TypeVar('R')

def function_a()-> int:
    return 5

def function_b()-> int:
    return 6

def execute_sync_tasks(tasks: Dict[str, Callable[[], R]]) -> List[Tuple[str, R]]:
    results = []
    for task_name, func in tasks.items():
        result = func()
        results.append((task_name, result))
    return results

res = execute_sync_tasks({'a': function_a, 'b': function_b})
res

[('a', 5), ('b', 6)]

In [13]:
import asyncio

async def async_task_one() -> int:
    await asyncio.sleep(0.1) 
    return 10 * 2

async def async_task_two() -> int:
    print("Running async task two")
    await asyncio.sleep(0.2)
    return 30 + 5


async def execute_async_tasks(tasks: Dict[str, Callable[[],Coroutine[Any, Any, R]]]) -> Tuple[str, R]:
    results = []
    for task_name, coro_func in tasks.items():
        result = await coro_func() 
        results.append((task_name, result))
    return results

res = await execute_async_tasks({'task_one':async_task_one, 'task_two': async_task_two})
res

Running async task two


[('task_one', 20), ('task_two', 35)]

In [ ]:
# when a function if the argument of another function. 

R = TypeVar('R')
P = ParamSpec('P')

async def call_async_task_with_args(
    async_task_func: Callable[P, Coroutine[Any, Any, R]],
    *args: P.args,
    **kwargs: P.kwargs
) -> R:
    coroutine = async_task_func(*args, **kwargs)
    result = await coroutine
    # same as result = await async_task_func(*args, **kwargs)
    
    return result

async def add_numbers(x: int, y: int) -> int:
    await asyncio.sleep(3)
    return x + y

result = await call_async_task_with_args(add_numbers, 3, 4)
result

7

In [ ]:
class Quackable(Protocol):
    def quack(self) -> str: ...
    sound_level: int
    
class Duck:
    def quack(self) -> str:
        return "Quack!"
    sound_level: int = 10
    
    
def make_it_quack(obj: Quackable) -> None:
    print(f"{obj.quack()} Level: {obj.sound_level}")
    
duckie = Duck()
   
make_it_quack(duckie)

# if you want to make it runtime checkable, e.g. with isinstance() and issubclass() then, wrap it with @typing.runtime_checkable

Quack! Level: 10


In [ ]:
class ExportTableRecord(Protocol):
    def get_id(self) -> str: ...
    def export_as_string(self) -> str: ...
    @property
    def record_type(self) -> str: ...
    

class ProductRecord:
    def __init__(self, product_id: str, name: str, price: float):
        self.product_id = product_id
        self.name = name
        self.price = price
        self._record_type = "PRODUCT" # Note: using _ for internal, exposed via property

    def get_id(self) -> str:
        return self.product_id

    def export_as_string(self) -> str:
        return f"Product: {self.name}, Price: ${self.price:.2f}"

    @property
    def record_type(self) -> str: # This makes it conform for record_type
        return self._record_type

class EventRecord:
    def __init__(self, event_uuid: str, timestamp: str, data: dict):
        self.event_uuid = event_uuid
        self.timestamp = timestamp
        self.data = data
        # For record_type, let's make it a simple attribute here
        self.record_type: str = "EVENT" # Direct attribute

    def get_id(self) -> str:
        return self.event_uuid

    def export_as_string(self) -> str:
        return f"Event at {self.timestamp} - Data: {self.data}"
    
    
def process_and_log_records(records: List[ExportTableRecord]): # What is the type of 'records'?
    print("--- Processing Records ---")
    for record in records:
        # How do you safely access methods/attributes defined in the protocol?
        record_id = record.get_id()
        exported_data = record.export_as_string()
        rec_type = record.record_type

        print(f"Type: {rec_type}, ID: {record_id}, Data: {exported_data}")
    print("--- Finished Processing ---")
    
events = [
    EventRecord("evt-abc", "2023-10-26T10:00:00Z", {"action": "login", "user": "user1"}),
    EventRecord("evt-def", "2023-10-26T10:05:00Z", {"action": "click", "item": "buttonA"})
]

products = [
    ProductRecord("P001", "Laptop", 1200.00),
    ProductRecord("P002", "Mouse", 25.00)
]

process_and_log_records(products)

--- Processing Records ---
Type: PRODUCT, ID: P001, Data: Product: Laptop, Price: $1200.00
Type: PRODUCT, ID: P002, Data: Product: Mouse, Price: $25.00
--- Finished Processing ---
